# Проект спринта 7

## Секреты Темнолесья

### Цель проекта



Подготовить данные об игровой индустрии за период с 2000 по 2013 год для последующего анализа. Изучить данные о продажах игр, игровых платформах, жанрах и оценках пользователей и критиков, выявить особенности развития рынка видеоигр и подготовить качественный срез данных для исследования.


### Задачи проекта

- Загрузить и изучить исходный датасет.
- Проверить данные на наличие ошибок и пропусков.
- Выполнить предобработку данных.
- Отобрать игры, выпущенные в период с 2000 по 2013 год включительно.
- Категоризовать игры по оценкам пользователей и критиков на группы: низкая, средняя и высокая оценка.
- Выделить наиболее популярные игровые платформы по количеству выпущенных игр.
- Подготовить данные для анализа продаж игр по платформам, жанрам и регионам.
- Исследовать особенности развития игровой индустрии в начале XXI века.

### Описание данных

Данные `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:


- `Name` — название игры.
- `Platform` — название платформы.
- `Year of Release` — год выпуска игры.
- `Genre` — жанр игры.
- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
- `EU sales` — продажи в Европе (в миллионах проданных копий).
- `JP sales` — продажи в Японии (в миллионах проданных копий).
- `Other sales` — продажи в других странах (в миллионах проданных копий).
- `Critic Score` — оценка критиков (от 0 до 100).
- `User Score` — оценка пользователей (от 0 до 10).
- `Rating` — рейтинг организации ESRB (*англ. Entertainment Software Rating Board*). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта


1. Загрузка и знакомство с данными.
2. Исследование структуры датасета.
2.1 Названия, или метки, столбцов датафрейма
2.2. Поиск и обработка пропущенных значений
2.3 Проверка и корректировка типов данных
2.4 Явные и неявные дубликаты в данных
3. Работа с датой выпуска игр.
3.1 Фильтрация данных за период 2000–2013 годов.
3.2 Категоризация игр по оценкам пользователей и критиков.
3.3 Определение топ-7 игровых платформ по количеству выпущенных игр.
4. Основной вывод по результатам побработки данных.

## Загрузка и знакомство с данными

In [1]:
#*Импортируем библиотеку pandas*
import pandas as pd

In [2]:
# Выгружаем данные из датасета new_games.csv в датафрейм df
df=pd.read_csv('new_games.csv')

In [3]:
# Выводим общую информацию о датафрейме
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15099 entries, 0 to 15098
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             15097 non-null  str    
 1   Platform         15099 non-null  str    
 2   Year of Release  14860 non-null  float64
 3   Genre            15097 non-null  str    
 4   NA sales         15099 non-null  float64
 5   EU sales         15099 non-null  str    
 6   JP sales         15099 non-null  str    
 7   Other sales      15099 non-null  float64
 8   Critic Score     7747 non-null   float64
 9   User Score       9417 non-null   str    
 10  Rating           9368 non-null   str    
dtypes: float64(4), str(7)
memory usage: 1.3 MB


Датасет `new_games.csv` содержит 11 столбцов и 15 099 строк, в которых представлена информация о видеоиграх, их жанрах, платформах, продажах в различных регионах мира и пользовательских оценках.

In [4]:
# Выводим первые строки датафрейма на экран
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


#### Изучим типы данных и их корректность:


**Числовые значения с плавающей запятой (float64)**. Четыре столбца имеют тип float64: `Year of Release`, `NA_sales`,`Other sales`, `Critic Score`. 
Столбцы `NA sales`, `Other sales` и `Critic Score` имеют корректный тип float64, поскольку содержат дробные числовые значения. 
Столбец `Year of Release` также представлен типом float64, однако год выпуска является целочисленным значением. 
Данный тип выбран из-за наличия пропусков, поэтому после их обработки столбец следует привести к типу int64.

**Строковые данные (object/str)**. Семь столбцов имеют строковый тип данных: `Name`, `Platform`, `Genre`, `EU sales`, `JP sales`, `User Score`, `Rating`.
Столбцы `Name`, `Platform`, `Genre` и `Rating` содержат текстовую информацию и имеют корректный тип данных.
Столбцы `EU sales` и `JP sales` содержат числовые значения продаж, однако представлены строковым типом данных. Для дальнейшего анализа их необходимо 
преобразовать в числовой формат.
Столбец `User Score` содержит пользовательские оценки, но также имеет строковый тип данных. Вероятно, в нём присутствуют нечисловые значения или пропуски, 
поэтому потребуется дополнительная обработка и преобразование в числовой формат.

**Пропущенные значения**

В датасете обнаружены пропуски:

* Year of Release - 239 пропусков;
* Name - 2 пропуска;
* Genre - 2 пропуска;
* Critic Score - 7 352 пропуска;
* User Score - 5 682 пропуска;
* Rating - 5 731 пропуск.

Наибольшее количество пропусков наблюдается в столбцах с оценками пользователей и критиков, а также возрастным рейтингом игр. Эти пропуски необходимо дополнительно
изучить и определить способ их обработки.

**Общий вывод**

После анализа структуры данных видно, что большая часть столбцов имеет корректные типы данных. Однако для проведения дальнейшего исследования потребуется:

1. обработать пропущенные значения;
2. преобразовать типы данных в столбцах `Year of Release`, `EU sales`, `JP sales` и `User Score`
3. привести названия столбцов к единому формату
4. проверить данные на наличие дубликатов и аномальных значений.

После выполнения предобработки данные будут готовы для анализа развития игровой индустрии в период с 2000 по 2013 год.

## Исследование структуры датасета

#### 1. Названия, или метки, столбцов датафрейма

In [5]:
# Выводим названия всех столбцов
df.columns


Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')

In [6]:
# Приводим названия столбцов к snake_case:
#нижний регистр, пробелы заменяем на подчёркивание
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')

In [7]:
# Вывод: названия столбцов приведены к единому стилю snake_case

#### 2. Поиск и обработка пропущенных значений

In [8]:

# Считаем количество пропусков в каждом столбце
df.isnull().sum()

name                  2
platform              0
year_of_release     239
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       7352
user_score         5682
rating             5731
dtype: int64

In [9]:
# Считаем долю пропусков в процентах
df.isnull().sum() / len(df) * 100

name                0.013246
platform            0.000000
year_of_release     1.582886
genre               0.013246
na_sales            0.000000
eu_sales            0.000000
jp_sales            0.000000
other_sales         0.000000
critic_score       48.691966
user_score         37.631631
rating             37.956156
dtype: float64


 В данных наблюдаются пропуски в следующих столбцах:
 
Столбец **'name'** — 2 пропусков (0.01%), возможная причина: *ошибка при сборе данных или отсутствие информации о названии игры в исходном источнике*

толбец **'year_of_release'** — 239 пропусков (1.58%), возможная причина: *для части игр не удалось определить или подтвердить год выпуска при 
формировании датасета.*

Столбец **'genre'** — 2 пропусков (0.01%), возможная причина: *отсутствует информация о жанре игры либо ошибка при объединении данных из разных источников.*

Столбец **'critic_score'** — 7352 пропусков (48.69%), возможная причина: *значительная часть игр была выпущена в ранние годы развития игровой индустрии, 
когда системы агрегирования профессиональных рецензий ещё не были широко распространены.*

Столбец **'user_score'** — 5682 пропусков (37.63%), возможная причина:* многие старые игры были выпущены до активного развития интернет-площадок с пользовательскими оценками.*

Столбец **'rating'** — 5731 пропусков (37.95%), возможная причина:*Географическое ограничение,выпущенных до создания организации ESRB*

In [10]:
# Удаляем строки где пропусков мало (название,жанр и год выпуска игры)
df=df.dropna(subset=['name', 'genre','year_of_release'])
df.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [11]:
# critic_score и user_score — оставляем пропуски, не трогаем
# Причина: отсутствие оценки — это факт, а не ошибка данных

In [12]:
# Заменяем пропуски в rating на строку-индикатор
df['rating'] = df['rating'].fillna('unknown')

#### 3. Проверка и корректировка типов данных

In [13]:
# Смотрим типы данных всех столбцов
df.dtypes 


name                   str
platform               str
year_of_release    float64
genre                  str
na_sales           float64
eu_sales               str
jp_sales               str
other_sales        float64
critic_score       float64
user_score             str
rating                 str
dtype: object

In [14]:
# Проверяем уникальные значения user_score — ищем строковые значения
df['user_score'].unique()

<StringArray>
[  '8',   nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4', '8.2',
   '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2', '8.9', '6.4',
 '7.8', '7.5', '2.6', '7.2', '9.2',   '7', '7.3', '4.3', '7.6', '5.7',   '5',
 '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8', '6.1', '6.7', '5.4',   '4',
 '4.9', '4.5', '9.3', '6.2', '4.2',   '6', '3.7', '4.1', '5.8', '5.6', '5.5',
 '4.4', '4.6', '5.9', '3.9', '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5',
 '2.5', '1.9',   '3', '2.7', '2.2',   '2', '9.5', '2.1', '3.6', '2.8', '1.8',
 '3.8',   '0', '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
 '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9',   '1', '9.7']
Length: 97, dtype: str

In [15]:
# user_score содержит строковые значения например 'tbd'
# Заменяем их на NaN и сразу переводим в числовой тип
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
df['user_score'].head()

0    8.0
1    NaN
2    8.3
3    8.0
4    NaN
Name: user_score, dtype: float64

In [16]:
# year_of_release — пропуски уже удалены на шаге 3, меняем тип на int
df['year_of_release'] = df['year_of_release'].astype('int64')
df.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006,Sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,NES,1985,Platform,29.08,3.58,6.81,0.77,NaN,NaN,unknown
2,Mario Kart Wii,Wii,2008,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009,Sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,unknown


In [17]:
# Проверяем eu_sales и jp_sales на наличие строковых данных
df[['eu_sales','jp_sales']].info()


<class 'pandas.DataFrame'>
Index: 14858 entries, 0 to 15098
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   eu_sales  14858 non-null  str  
 1   jp_sales  14858 non-null  str  
dtypes: str(2)
memory usage: 348.2 KB


In [18]:
# Переводим оба столбца из str в числовой формат float64
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')


In [19]:
# Проверим итоговые типы данных
df.dtypes

name                   str
platform               str
year_of_release      int64
genre                  str
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating                 str
dtype: object

#### 4. Явные и неявные дубликаты в данных




In [20]:
# Смотрим уникальные значения в столбце genre
df['genre'].unique()

<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy',
         'MISC', 'ROLE-PLAYING',       'RACING',       'ACTION',
      'SHOOTER',     'FIGHTING',       'SPORTS',     'PLATFORM',
    'ADVENTURE',   'SIMULATION',       'PUZZLE',     'STRATEGY']
Length: 24, dtype: str

In [21]:
# Смотрим уникальные значения в столбце platform
df['platform'].unique()

<StringArray>
[ 'Wii',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  'PS4',  '3DS',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XOne',
 'WiiU',   'GC',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str

In [22]:
# Смотрим уникальные значения в столбце rating
df['rating'].unique()

<StringArray>
['E', 'unknown', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str

In [23]:
# Приводим жанры и платформы к нижнему регистру
df['genre'] = df['genre'].str.lower()
df['platform'] = df['platform'].str.lower()
df.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,nes,1985,platform,29.08,3.58,6.81,0.77,NaN,NaN,unknown
2,Mario Kart Wii,wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,gb,1996,role-playing,11.27,8.89,10.22,1.00,NaN,NaN,unknown


In [24]:
# Приводим рейтинг к верхнему регистру
df['rating'] = df['rating'].str.upper()
df.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
1,Super Mario Bros.,nes,1985,platform,29.08,3.58,6.81,0.77,NaN,NaN,UNKNOWN
2,Mario Kart Wii,wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
4,Pokemon Red/Pokemon Blue,gb,1996,role-playing,11.27,8.89,10.22,1.00,NaN,NaN,UNKNOWN


In [25]:
# Проверяем явные дубликаты — считаем количество
df.duplicated().sum()

np.int64(199)

In [26]:
# Удаляем явные дубликаты
df = df.drop_duplicates().reset_index(drop=True)

In [27]:
#проверяем удаление дубликатов
df.duplicated().sum()

np.int64(0)

 **Промежуточный вывод по дубликатам:**
 В ходе проверки категориальных столбцов была проведена нормализация данных:
- жанры и платформы приведены к нижнему регистру, рейтинг — к верхнему.
- Неявных дубликатов, связанных с опечатками или разным написанием, не обнаружено.
- Явных дубликатов найдено: 199.
- Все 199 дублирующихся строк удалены методом drop_duplicates().
- После удаления дубликатов датафрейм содержит 14 659 строк.

 **Общий промежуточный вывод по предобработке данных:**
 В процессе подготовки данных были выполнены следующие действия:
- удалены строки с пропусками в столбцах name, genre и year_of_release (241 строка);
- пропуски в столбце rating заменены на значение-индикатор 'UNKNOWN';
- пропуски в столбцах critic_score и user_score оставлены как есть,
  так как отсутствие оценки является фактом, а не ошибкой данных;
- приведены типы данных: eu_sales, jp_sales и user_score — к float64,
  year_of_release — к int64;
- удалены 199 явных дубликатов.
 Итого удалено 440 строк. Итоговый датафрейм содержит 14 659 строк.

## Работа с датой выпуска игр

#### 1. Фильтрация данных


In [28]:
# Отбираем игры за период с 2000 по 2013 год включительно
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)]
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
2,Mario Kart Wii,wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
6,New Super Mario Bros.,ds,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E
7,Wii Play,wii,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E


#### 2. Категоризация данных

Категоризируем игры по оценкам пользователей:

In [29]:
# высокая оценка: от 8 до 10 включительно
# средняя оценка: от 3 до 8, не включая 8
# низкая оценка: от 0 до 3, не включая 3
def user_score_category(score):
    if score >= 8:
        return 'высокая оценка'
    elif score >= 3:
        return 'средняя оценка'
    else:
        return 'низкая оценка'

df_actual['user_score_category'] = df_actual['user_score'].apply(user_score_category)


Категоризируем игры по оценкам критиков:

In [30]:
# высокая оценка: от 80 до 100 включительно
# средняя оценка: от 30 до 80, не включая 80
# низкая оценка: от 0 до 30, не включая 30
def critic_score_category(score):
    if score >= 80:
        return 'высокая оценка'
    elif score >= 30:
        return 'средняя оценка'
    else:
        return 'низкая оценка'

df_actual['critic_score_category'] = df_actual['critic_score'].apply(critic_score_category)

#### 3. Определение топ-7 игровых платформ

In [31]:
# Выделяем топ-7 платформ по количеству выпущенных игр за актуальный период
top_7_platforms = df_actual['platform'].value_counts().head(7)
top_7_platforms

platform
ps2     1929
ds      1856
wii     1179
x360    1058
ps3     1037
psp      947
xb       736
Name: count, dtype: int64

## 4. Основной вывод по результатам предобработки данных



 В ходе работы был подготовлен датасет new_games.csv,
 содержащий информацию о продажах видеоигр разных жанров и платформ.

 Прдобработка данных:
 — названия столбцов приведены к единому стилю snake_case;
 — удалены строки с пропусками в столбцах name, genre и year_of_release (241 строка);
 — пропуски в столбце rating заменены на значение-индикатор 'UNKNOWN';
 — пропуски в critic_score и user_score оставлены: отсутствие оценки — это факт;
 — исправлены типы данных: eu_sales, jp_sales, user_score приведены к float64,
   year_of_release — к int64;
 — жанры и платформы приведены к нижнему регистру,
   рейтинг — к верхнему регистру;
 — удалены 199 явных дубликатов.
 Итого удалено 440 строк из 15 099. Осталось 14 659 строк.

 Описание среза данных:
 Для анализа отобраны игры за период с 2000 по 2013 год включительно.
 Срез сохранён в датафрейм df_actual.
 Срез содержит 11381 строк.

 Новые поля добавленные в датасет:
 — user_score_category: категория оценки пользователей
   (высокая оценка: 8–10, средняя оценка: 3–8, низкая оценка: 0–3);
 — critic_score_category: категория оценки критиков
   (высокая оценка: 80–100, средняя оценка: 30–80, низкая оценка: 0–30).

 ТОП-7 Платформ по количеству выпущенных игр за период 2000–2013:
 1. PS2  — 1929 игр
 2. DS   — 1856 игр
 3. Wii  — 1179 игр
 4. X360 — 1058 игр
 5. PS3  — 1037 игр
 6. PSP  —  947 игр
 7. XB   —  736 игр

 Подготовленный датасет готов для дальнейшего анализа
 продаж игр по платформам, жанрам и регионам.